# 07 - AI Smart Config

> **When to use**: When you don't want to hand-write YAML configs and want AI to auto-analyze table structure and generate optimal configs.
>
> **Core concept**: The `sqlseed-ai` plugin uses an LLM to analyze schema semantics, auto-generating configs with a self-correction loop.

## Applicable Scenarios

- Complex table structure, don't want to hand-write config → AI auto-generates
- Unsure which generator to use → AI recommends based on column name semantics
- Need rapid prototyping → AI generates config with one command

## What You Will Learn

- SchemaAnalyzer workflow
- AiConfigRefiner self-correction loop
- Configured backend and local model registry
- ErrorSummary error classification
- Caching mechanism

Offline cells need no model service. Live inference is disabled by default; configure your backend and explicitly enable the live cell to run it. A skipped live cell is not LLM acceptance.

**📚 Tutorial Navigation**

| No. | Topic | Architecture Layer | Prerequisites |
|------|------|--------|----------|
| 01 | Quick Start and Core Workflow | Orchestrator | None |
| 02 | 9-Level Strategy Chain | Core: ColumnMapper | 01 |
| 03 | Generators and Provider System | Generators | 01 |
| 04 | Database Layer and Multi-table | Database + Core | 01 |
| 05 | Expression Derivation and Constraint Solving | Core: DAG / Expression | 01 |
| 06 | Config-Driven and Transform | Config / Core | 01 |
| **→ 07** | **AI Smart Config** | **Plugins: AI** | **01** |
| 08 | MCP Server Integration | Plugins: MCP | 01 |
| 09 | Plugin System and Hook Lifecycle | Plugins | 01 |
| 10 | CLI Reference Manual | CLI | 06 |
| 11 | Utilities Reference | Utils | 01 |
| 12 | Testing Integration Patterns | Testing | 01 |

---

In [ ]:
from __future__ import annotations

# Run from examples/notebooks. Install from the repository root in one resolution:
# python -m pip install -e ".[dev,all]" -e "./plugins/sqlseed-cli" \
#   -e "./plugins/sqlseed-ai[dev,mcp]" -e "./plugins/mcp-server-sqlseed" -e "./plugins/sqlseed-web[dev]"
import os
import sqlite3
import sys
import tempfile
from contextlib import closing
from pathlib import Path

import sqlseed
from sqlseed import connect, fill_from_config

sys.path.insert(0, str(Path("..").resolve()))  # build_demo_db only
from build_demo_db import build

# Keep this object alive across cells. No existing database is opened or rebuilt.
_demo_directory = tempfile.TemporaryDirectory(prefix="sqlseed-notebook-")
demo_root = Path(_demo_directory.name)
os.environ["SQLSEED_CACHE_DIR"] = str(demo_root / "cache")
db_path = build(demo_root / "demo.db")


def require(condition, message):
    """Stop the tutorial if an expected outcome did not occur."""
    if not condition:
        raise RuntimeError(message)


def check_generation(result, count):
    """Check errors and generated row count before showing success."""
    require(not result.errors and result.count == count, f"Generation failed: {result.errors}; count={result.count}")


def read_rows(database, sql):
    """Read actual persisted values using a fixed tutorial query."""
    # The queries below are fixed tutorial SQL, never external identifiers.
    with closing(sqlite3.connect(database)) as connection:
        return connection.execute(sql).fetchall()


with connect(str(db_path), provider="faker") as orch:
    for table, count in (("organizations", 5), ("members", 20), ("projects", 10), ("tags", 8)):
        check_generation(orch.fill_table(table, count=count, seed=42, skip_ai=True), count)

print(f"sqlseed {sqlseed.__version__} | Temporary database: {db_path}")

### 📍 Architecture Position

| Module | File | Core Class/Function |
|------|------|------------|
| Schema Analysis | `plugins/sqlseed-ai/src/sqlseed_ai/analyzer.py` | `SchemaAnalyzer` |

> Corresponding architecture diagram: [§7 AI Plugin Architecture](../../docs/architecture.zh-CN.md#7-ai-插件架构)

## 1. See It in Action — AI Auto-Analyzes Table Structure

Give sqlseed-ai a table name and it will auto-analyze the schema and generate the optimal config — **no hand-written YAML needed**:

```
Input: organizations table
Output: auto-generated YAML config (including generator, params, constraints)
```

Below we first verify the plugin installation, then demo the full workflow.

## 2. sqlseed-ai Installation Verification

In [ ]:
import sqlseed_ai

print(f"AI plugin imported: {sqlseed_ai.__name__}; live inference remains disabled.")

## 3. SchemaAnalyzer Workflow

SchemaAnalyzer collects the full table context (columns, indexes, FKs, sample data), sends it to the LLM for analysis, and returns a YAML config.

In [ ]:
from sqlseed_ai.analyzer import SchemaAnalyzer

from sqlseed import connect

with connect(str(db_path)) as orch:
    schema_ctx = orch.get_schema_context("organizations")
    print("Schema context keys:", list(schema_ctx.keys()))
    print(f"Columns: {len(schema_ctx['columns'])}")
    print(f"Foreign keys: {len(schema_ctx['foreign_keys'])}")
    print(f"Indexes: {len(schema_ctx['indexes'])}")
    print(f"Sample data: {len(schema_ctx['sample_data'])} rows")
    print(f"All tables: {schema_ctx['all_table_names']}")

## 4. AiConfigRefiner Self-Correction Loop

`AiConfigRefiner` validates a generated configuration and requests repairs. This cell makes a real model call only after you set `RUN_LIVE_AI = True`.

Configure `SQLSEED_AI_BACKEND`, `SQLSEED_AI_BASE_URL` and `SQLSEED_AI_MODEL` for your service. Google AI Studio requires explicit `SQLSEED_AI_BACKEND=google_ai_studio` (or its recognized URL), plus `SQLSEED_AI_API_KEY` / `GOOGLE_API_KEY`. A key alone does not select Google. Local LM Studio / Ollama generally need no API key. See the [AI guide](../../docs/gemma4-integration.md).

Failures in the live branch propagate; the offline configuration example below is not model output.


In [ ]:
from sqlseed_ai.config import AIConfig

RUN_LIVE_AI = False  # Explicitly opt in after configuring a real backend.

if RUN_LIVE_AI:
    from sqlseed_ai.refiner import AiConfigRefiner

    backend_config = AIConfig.from_env()
    analyzer = SchemaAnalyzer(config=backend_config)
    refiner = AiConfigRefiner(analyzer, db_path=str(db_path), cache_dir=str(demo_root / "ai-cache"))
    generated_columns = refiner.generate_and_refine("organizations")
    require(isinstance(generated_columns, dict) and bool(generated_columns), "Model returned no configuration")
    print("Live model produced a configuration; review it before filling:")
    print(generated_columns)
else:
    print("Live LLM inference: NOT RUN. Offline configuration validation follows separately.")

## 5. Model Registry

The installed plugin maintains a Gemma model priority list. Listing this registry makes no network request and does not prove model availability or pricing; actual selection depends on the configured backend and its response.


In [ ]:
from sqlseed_ai._model_selector import _GEMMA_MODEL_PRIORITY

print(f"Installed Gemma registry ({len(_GEMMA_MODEL_PRIORITY)} entries):")
for model in _GEMMA_MODEL_PRIORITY:
    print(f"  {model.value}: {model.display_name}")

## 6. ErrorSummary Classification

`summarize_error()` classifies by exception type and message. Different Python exception classes may share `runtime_error`; the number of examples below is not a count of distinct categories.


In [ ]:
from sqlseed_ai.errors import summarize_error

test_errors = [
    ValueError("count must be greater than 0"),
    TypeError("'NoneType' object is not iterable"),
    KeyError("missing_column"),
    ImportError("No module named faker"),
    FileNotFoundError("/nonexistent.db"),
    PermissionError("read-only database"),
    RuntimeError("unexpected error"),
]

print("ErrorSummary for representative exceptions:")
for err in test_errors:
    summary = summarize_error(err)
    print(f"  {type(err).__name__:20s} -> {summary.error_type}")

## 7. Offline Configuration → Validation → Fill

This is a hand-written example in the format used by reviewed AI suggestions, not a simulated successful model response. The pattern generator takes `regex`. Foreign-key values are left to Core's relation handling. Use an independent, empty database so clearing organizations cannot break the populated members/projects in the earlier database.


In [ ]:
import re

from sqlseed.config.loader import save_config
from sqlseed.config.models import ColumnConfig, GeneratorConfig, TableConfig

ORG_PATTERN = r"ORG-\d{4}"
config_db = build(demo_root / "reviewed-config.db")
reviewed_config = GeneratorConfig(
    db_path=str(config_db),
    provider="faker",
    tables=[
        TableConfig(
            name="organizations",
            count=3,
            clear_before=True,
            seed=42,
            columns=[
                ColumnConfig(name="org_code", generator="pattern", params={"regex": ORG_PATTERN}),
                ColumnConfig(name="name", generator="company"),
                ColumnConfig(name="description", generator="sentence"),
            ],
        )
    ],
)
config_path = demo_root / "reviewed-config.yaml"
save_config(reviewed_config, str(config_path))
results = fill_from_config(str(config_path), skip_ai=True)
require(len(results) == 1, "Expected one target table")
check_generation(results[0], 3)
rows = read_rows(config_db, "SELECT org_code, name FROM organizations ORDER BY org_code")
require(len(rows) == 3, "Expected three persisted organizations")
require(all(re.fullmatch(ORG_PATTERN, row[0]) for row in rows), "Pattern values do not match the configured regex")
print(rows)

## Summary

| Feature | Description |
|------|------|
| SchemaAnalyzer | Collects table context, sends to LLM |
| AiConfigRefiner | Self-correction loop: generate→validate→fix |
| Model Registry | Inspect installed model IDs; live availability is separate |
| ErrorSummary | Typed/message-based error categories |
| File Cache | Platform-standard cache dir |

**Next**: [08-mcp-server.ipynb](08-mcp-server.ipynb) — MCP Server Integration

In [ ]:
require(
    len(read_rows(config_db, "SELECT org_code FROM organizations")) == 3, "Reviewed config did not persist three rows"
)
require(
    len(read_rows(db_path, "SELECT member_id FROM members")) == 20, "The independent example changed existing members"
)
print("Offline configuration and SQLite validation passed; live LLM status is reported in cell 4.")